In [3]:
# importing library
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [5]:
# importing data from excel file
df = pd.read_excel('E:\script.xlsx')

In [6]:
# overview data
df.head()

,date,date_time,day_of_week,first_scan,confirm_at,route _code,bks,truck_weight,shift,user_standard,number_bag,processing_time,number_user,processing_speed
0,20230731,2023-07-31,Monday,2023-07-31 05:00:08,2023-07-31 07:00:08,A,29,1.00,6-8h,1,165,120,2,30
1,20230801,2023-08-01,Tuesday,2023-08-01 06:00:09,2023-08-01 06:40:09,A,29,1.00,6-8h,1,157,40,2,25
2,20230802,2023-08-02,Wednesday,2023-08-02 03:00:06,2023-08-02 03:57:06,A,29,1.00,0-6h,1,112,57,1,30
3,20230803,2023-08-03,Thursday,2023-08-03 07:00:58,2023-08-03 08:00:58,B,30,1.25,8-10h,2,107,60,2,17
4,20230804,2023-08-04,Friday,2023-08-04 06:00:07,2023-08-04 09:00:07,B,88,1.90,8-10h,2,186,180,2,15


In [7]:
# converting data type to the right type
df['first_scan'] = pd.to_datetime(df['first_scan'])
df['confirm_at'] = pd.to_datetime(df['confirm_at'])

In [15]:
# estimating the number of employees who can complete the job
def estimate_user(df_source):
    
    df_est = df_source.copy()
    df_est = df_est.sort_values(by=['first_scan']).reset_index(drop=True)
    
    df_est['needed_user'] = np.nan
    df_est.loc[0, 'needed_user'] = 0
    df_est['surplus_user'] = np.nan
    df_est.loc[0, 'surplus_user'] = df_est.loc[0, 'user_standard']

    for i in range(1, len(df_est)):
        df_est.loc[i, 'needed_user'] = 0
        overlap_arr = np.array([])
        for j in range(i):
            if df_est.loc[i, 'first_scan'] < df_est.loc[j, 'confirm_at']:
                overlap_arr = np.append(overlap_arr, df_est.loc[j, 'user_standard'])
            else:
                overlap_arr = np.append(overlap_arr, 0)

        df_est.loc[i, 'needed_user'] = np.sum(df_est.loc[:i, 'surplus_user']) - np.sum(overlap_arr)
        if df_est.loc[i, 'needed_user'] < df_est.loc[i, 'user_standard']:
            df_est.loc[i, 'surplus_user'] = df_est.loc[i, 'user_standard'] - df_est.loc[i, 'needed_user']
        else:
            df_est.loc[i, 'surplus_user'] = 0
            
    return df_est

In [16]:
partition_date = df['date_time'].unique()
partition_shift = df['shift'].unique()

In [17]:
# Checking by shift
final_df_shift = pd.DataFrame()

for partition in partition_shift:
    df_tmp = df[df['shift'] == partition]
    df_tmp = estimate_user(df_tmp)
    
    final_df_shift = pd.concat([final_df_shift, df_tmp])

In [18]:
final_df_shift.sort_values(by='date').to_excel('export_with_partition_shift.xlsx')